# Experiment 4: Building a Retrieval-Augmented Generation (RAG) Pipeline with LangChain

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To design and implement an end-to-end Retrieval-Augmented Generation (RAG) pipeline using LangChain and ChromaDB that answers user questions from a set of PDF documents, and to evaluate it using RAGAs metrics (faithfulness, answer relevancy, context precision and context recall).

## 2. Objectives

- To understand why an LLM alone cannot answer questions about private or recent documents.
- To load PDF documents, split them into chunks and convert the chunks into embeddings.
- To store the embeddings in a ChromaDB vector store and retrieve the most similar chunks for a question.
- To build a LangChain chain that combines retrieval, a prompt and an LLM.
- To evaluate the pipeline with RAGAs and compare two retrieval settings.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- API key for the LLM (Gemini through the OpenAI SDK is used here; any OpenAI-compatible model works)
- Required Python libraries: `langchain`, `langchain-community`, `langchain-openai`, `langchain-huggingface`, `langchain-chroma`, `chromadb`, `pypdf`, `sentence-transformers`, `ragas`, `pandas`

## 4. Theory

### A. Retrieval-Augmented Generation

A language model only knows what it saw during training, and it may invent ("hallucinate") facts it does not know. RAG fixes this by looking up relevant text first and placing it in the prompt:

`question -> retrieve relevant chunks -> prompt = chunks + question -> LLM -> grounded answer`

### B. Components of the pipeline

| Component | Purpose |
|---|---|
| Document loader | Reads the PDF files into text with page numbers. |
| Text splitter | Cuts the text into overlapping chunks small enough to embed and to fit in a prompt. |
| Embedding model | Converts each chunk into a vector so that similar meanings are close together. |
| Vector store (ChromaDB) | Stores the vectors and returns the nearest chunks to a question. |
| Prompt and LLM | Answer the question using only the retrieved chunks. |

### C. RAGAs evaluation metrics

RAGAs uses an LLM as a judge to score a RAG pipeline without hand-labelling every answer.

| Metric | Question it answers | Needs |
|---|---|---|
| Faithfulness | Is every claim in the answer supported by the retrieved chunks? | question, answer, chunks |
| Answer relevancy | Does the answer address the question asked? | question, answer |
| Context precision | Are the relevant chunks ranked near the top? | question, reference answer, chunks |
| Context recall | Do the retrieved chunks contain the information in the reference answer? | question, reference answer, chunks |

All scores lie between 0 and 1; higher is better.

## 5. Algorithm / Procedure

1. Install and import the libraries; download the two research papers (RAG and LoRA) as PDF.
2. Load the PDFs and split them into chunks.
3. Embed the chunks and store them in ChromaDB.
4. Retrieve the chunks for a question and inspect them.
5. Build the RAG chain (retriever, prompt, LLM) and answer questions with sources.
6. Prepare a small evaluation set of questions with reference answers.
7. Run the pipeline for two retrieval settings (k = 4 and k = 8).
8. Score both with RAGAs and record observations.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q langchain langchain-community langchain-openai langchain-huggingface langchain-chroma chromadb pypdf sentence-transformers ragas pandas

### Step 2: Import the libraries and download the documents

In [2]:
import os, sys, types, json, asyncio, warnings, urllib.request
import pandas as pd
from dotenv import load_dotenv, find_dotenv
warnings.filterwarnings("ignore")
load_dotenv(find_dotenv(usecwd=True))   # reads GEMINI_API_KEY from the .env file

# download the two papers if they are not already in the data folder
os.makedirs("data", exist_ok=True)
papers = {"rag_lewis_2020.pdf": "https://arxiv.org/pdf/2005.11401", "lora_hu_2021.pdf": "https://arxiv.org/pdf/2106.09685"}
for name, url in papers.items():
    if not os.path.exists(f"data/{name}"):
        urllib.request.urlretrieve(url, f"data/{name}")
print(os.listdir("data"))

['lora_hu_2021.pdf', 'rag_lewis_2020.pdf']


### Step 3: Load the PDFs and split them into chunks

`PyPDFLoader` returns one document per page. `RecursiveCharacterTextSplitter` cuts the pages into chunks of about 800 characters; `chunk_overlap=100` repeats the end of a chunk at the start of the next so that a sentence cut at the boundary is not lost.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = []
for name in papers:
    pages += PyPDFLoader(f"data/{name}").load()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(pages)
print(len(pages), "pages ->", len(chunks), "chunks")
print(chunks[10].page_content[:300])

45 pages -> 237 chunks
For knowledge-intensive generation, we experiment with MS-MARCO [1] and Jeopardy question
generation, and we ﬁnd that our models generate responses that are more factual, speciﬁc, and
diverse than a BART baseline. For FEVER [56] fact veriﬁcation, we achieve results within 4.3% of
state-of-the-art pi


### Step 4: Embed the chunks and store them in ChromaDB

The `all-MiniLM-L6-v2` model turns every chunk into a 384-number vector. Chroma stores the vectors on disk and compares them with cosine similarity.

In [4]:
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
shutil.rmtree("chroma_db", ignore_errors=True)
store = Chroma.from_documents(chunks, embedder, collection_name="rag_papers", persist_directory="chroma_db",
                              collection_metadata={"hnsw:space": "cosine"})
print("Chunks stored:", store._collection.count())

Chunks stored: 237


### Step 5: Retrieve the chunks for a question

The retriever embeds the question and returns the `k` most similar chunks.

In [5]:
question = "Which weight matrices does LoRA adapt in most of its experiments?"
retriever = store.as_retriever(search_kwargs={"k": 4})
docs = retriever.invoke(question)

pd.DataFrame({"Rank": range(1, 5), "File": [d.metadata["source"].split("/")[-1] for d in docs],
              "Page": [d.metadata["page"] + 1 for d in docs], "Text": [d.page_content[:90].replace("\n", " ") for d in docs]})

,Rank,File,Page,Text
0,1,lora_hu_2021.pdf,6,is the number of adapter layers and ˆLLN the n...
1,2,lora_hu_2021.pdf,4,a subset of the pre-trained parameters. LoRA t...
2,3,lora_hu_2021.pdf,10,to maximize downstream performance? 2) Is the ...
3,4,lora_hu_2021.pdf,21,"First of all, LoRA+PE signiﬁcantly outperforms..."


### Step 6: Build the RAG chain

The chain is written with LangChain's pipe syntax: the retriever fills `context`, the prompt combines it with the question, the LLM answers, and a parser returns plain text. The prompt tells the model to use only the context and to say so when the answer is missing.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gemini-2.5-flash-lite", temperature=0, api_key=os.environ["GEMINI_API_KEY"],
                 base_url="https://generativelanguage.googleapis.com/v1beta/openai/", max_retries=4)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context. If the context does not contain the answer, reply exactly: "
               "I don't know based on the provided documents. Answer in at most three sentences."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")])

format_docs = lambda docs: "\n\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(docs, 1))
chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()

print(chain.invoke(question))

In most experiments, LoRA is applied only to Wq and Wv. LoRA considers weight matrices in the self-attention module.


### Step 7: Answer questions that are in the documents and one that is not

A good RAG system answers from the papers and refuses when the papers do not contain the answer.

In [7]:
for q in ["Which generator model does RAG use and how many parameters does it have?",
          "What is the price per million tokens of the GPT-4 API?"]:
    print("Q:", q, "\nA:", chain.invoke(q), "\n")

Q: Which generator model does RAG use and how many parameters does it have? 
A: The RAG models use BART-large as the generator, which has 406M parameters. 



Q: What is the price per million tokens of the GPT-4 API? 
A: I don't know based on the provided documents. 



### Step 8: Prepare the evaluation set and run the pipeline for k = 4 and k = 8

Each question has a reference answer written from the paper. The helper `run_rag(k)` retrieves the chunks and generates an answer for every question.

In [8]:
qa = [
 ("Which retriever does the RAG model use, and what architecture is it?", "RAG uses Dense Passage Retriever (DPR), a bi-encoder with a BERT-base document encoder and a BERT-base query encoder."),
 ("Which generator model does RAG use and how many parameters does it have?", "RAG uses BART-large, a pre-trained seq2seq transformer with 400M parameters, as its generator."),
 ("How was the Wikipedia knowledge source prepared for the RAG index?", "The December 2018 Wikipedia dump was split into disjoint 100-word chunks, giving 21M documents, which were embedded and stored in a FAISS MIPS index with an HNSW approximation."),
 ("What search problem is solved to find the top-K documents in RAG?", "Finding the top-K documents is a Maximum Inner Product Search (MIPS) problem, which can be approximately solved in sub-linear time."),
 ("By how much does LoRA reduce trainable parameters and GPU memory for GPT-3 175B compared with fine-tuning with Adam?", "LoRA reduces the number of trainable parameters by 10,000 times and the GPU memory requirement by 3 times."),
 ("Which weight matrices does LoRA adapt in most of its experiments?", "In most experiments LoRA is applied only to the query and value projection matrices, Wq and Wv, of the self-attention module."),
 ("Why does LoRA add no inference latency?", "After training, W = W0 + BA can be computed and stored explicitly, so inference runs on a single merged matrix exactly as in the original model."),
 ("How small can the LoRA rank be according to the rank experiments?", "A rank as small as one suffices for adapting both Wq and Wv on WikiSQL and MultiNLI, while training Wq alone needs a larger rank."),
]

def run_rag(k):
    rows = []
    for q, ref in qa:
        ctx = store.as_retriever(search_kwargs={"k": k}).invoke(q)
        ans = (prompt | llm | StrOutputParser()).invoke({"context": format_docs(ctx), "question": q})
        rows.append({"k": k, "question": q, "reference": ref, "answer": ans, "contexts": [d.page_content for d in ctx]})
    return rows

runs = run_rag(4) + run_rag(8)
print(len(runs), "answers generated")

16 answers generated


### Step 9: Score the answers with RAGAs

The four RAGAs metrics use the same Gemini model as the judge. The scoring functions are asynchronous, so a notebook can call them with `await`.

In [9]:
# compatibility shim: some ragas versions import optional Vertex AI classes that may not be installed
import langchain_community.llms as _llms
try:
    from langchain_community.chat_models.vertexai import ChatVertexAI
except ImportError:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules[_stub.__name__] = _stub
    _llms.VertexAI = type("VertexAI", (), {})

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings as RagasEmbeddings
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextPrecisionWithReference, ContextRecall

client = AsyncOpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/", max_retries=5)
judge = llm_factory("gemini-2.5-flash-lite", client=client)
faith = Faithfulness(llm=judge)
relev = AnswerRelevancy(llm=judge, embeddings=RagasEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2"))
prec = ContextPrecisionWithReference(llm=judge)
recall = ContextRecall(llm=judge)

In [10]:
sem = asyncio.Semaphore(6)   # limit parallel judge calls

async def score(r):
    async with sem:
        async def safe(coro):
            try:
                return (await coro).value
            except Exception:
                return float("nan")
        f, a, p, c = await asyncio.gather(
            safe(faith.ascore(user_input=r["question"], response=r["answer"], retrieved_contexts=r["contexts"])),
            safe(relev.ascore(user_input=r["question"], response=r["answer"])),
            safe(prec.ascore(user_input=r["question"], reference=r["reference"], retrieved_contexts=r["contexts"])),
            safe(recall.ascore(user_input=r["question"], reference=r["reference"], retrieved_contexts=r["contexts"])))
        return {"k": r["k"], "faithfulness": f, "answer_relevancy": a, "context_precision": p, "context_recall": c}

scores = pd.DataFrame(await asyncio.gather(*[score(r) for r in runs]))
scores.groupby("k").mean().round(2)

,faithfulness,answer_relevancy,context_precision,context_recall
k,,,,
4,0.88,0.51,0.39,0.50
8,0.88,0.62,0.43,0.62


## 7. Experimental Observations

### Observation Table 1: Retrieved chunks for the sample question

In [11]:
pd.DataFrame({"Rank": range(1, 5), "File": [d.metadata["source"].split("/")[-1] for d in docs],
              "Page": [d.metadata["page"] + 1 for d in docs], "Text": [d.page_content[:90].replace("\n", " ") for d in docs]}).set_index("Rank")

,File,Page,Text
Rank,,,
1,lora_hu_2021.pdf,6,is the number of adapter layers and ˆLLN the n...
2,lora_hu_2021.pdf,4,a subset of the pre-trained parameters. LoRA t...
3,lora_hu_2021.pdf,10,to maximize downstream performance? 2) Is the ...
4,lora_hu_2021.pdf,21,"First of all, LoRA+PE signiﬁcantly outperforms..."


### Observation Table 2: RAGAs scores (average over 8 questions)

In [12]:
scores.groupby("k").mean().round(2).rename(index={4: "k = 4", 8: "k = 8"})

,faithfulness,answer_relevancy,context_precision,context_recall
k,,,,
k = 4,0.88,0.51,0.39,0.50
k = 8,0.88,0.62,0.43,0.62


### Observation Table 3: Per-question scores for k = 4

In [13]:
detail = scores[scores.k == 4].drop(columns="k").round(2).reset_index(drop=True)
detail.insert(0, "Question", [q[:55] for q, _ in qa])
detail

,Question,faithfulness,answer_relevancy,context_precision,context_recall
0,"Which retriever does the RAG model use, and wh...",1.0,0.00,0.00,0.0
1,Which generator model does RAG use and how man...,1.0,0.86,0.25,0.0
2,How was the Wikipedia knowledge source prepare...,1.0,0.68,0.00,0.0
3,What search problem is solved to find the top-...,1.0,0.38,0.25,1.0
4,By how much does LoRA reduce trainable paramet...,0.0,0.00,0.00,0.0
5,Which weight matrices does LoRA adapt in most ...,1.0,0.78,0.83,1.0
6,Why does LoRA add no inference latency?,1.0,0.93,1.00,1.0
7,How small can the LoRA rank be according to th...,1.0,0.48,0.75,1.0


**Observations**

- The two papers (45 pages) were split into 237 chunks of about 800 characters and stored in ChromaDB. For the sample question the retriever returned four chunks from the LoRA paper, and the chain answered correctly: LoRA is applied to the query and value matrices, Wq and Wv.
- The chain answered a question from the papers (BART-large as the generator) and refused the question that the papers do not contain (the price of the GPT-4 API) with "I don't know based on the provided documents", instead of inventing a number.
- Faithfulness was 0.88 for both settings, so the model rarely added claims that were not in the retrieved text. Only the LoRA memory question scored 0, because the model answered that the context did not contain the answer.
- Retrieving more chunks helped the retrieval metrics: context recall rose from 0.50 (k = 4) to 0.62 (k = 8), context precision from 0.39 to 0.43, and answer relevancy from 0.51 to 0.62. A larger k gives the needed passage a better chance to be included, at the cost of a longer prompt.
- The first and fifth questions in Table 3 (the DPR retriever and the LoRA memory saving) show the weak point. When the chunk with the answer was not among the retrieved chunks, the model refused, and answer relevancy and context recall were 0. The generator question has recall 0 for a different reason: the paper states 406M parameters while the reference answer says 400M, so the judge counted it as unsupported. The retriever, not the LLM, is the usual bottleneck, so chunk size, overlap, the embedding model and k are the parameters to tune.

## 8. Result

A RAG pipeline was built with LangChain, a MiniLM embedding model and ChromaDB over two research papers (237 chunks). It answered questions from the documents and refused a question that was not in them. Evaluation with RAGAs on 8 questions gave faithfulness 0.88, answer relevancy 0.51, context precision 0.39 and context recall 0.50 for k = 4, and faithfulness 0.88, answer relevancy 0.62, context precision 0.43 and context recall 0.62 for k = 8. Retrieving more chunks improved the retrieval and relevancy scores without reducing faithfulness.

## 9. Conclusion

A RAG pipeline grounds a language model in the user's own documents: the PDFs are split, embedded and stored in ChromaDB, and the retrieved chunks are placed in the prompt so that the answers are based on the documents and can be traced to their pages. The RAGAs metrics measure the retrieval quality (context precision and recall) separately from the generation quality (faithfulness and answer relevancy), which shows where a pipeline should be improved, for example by changing the chunk size or the number of retrieved chunks.